[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-poly-reg.ipynb)

# Polynomial Regression

*AIBits Academy · Machine Learning End To End · Supervised Learning*

Capturing non-linear patterns by transforming features into higher-degree terms — still linear in parameters, non-linear in inputs.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> A straight line goes up or down at a fixed rate — it can't *bend*. But crop yield climbs with rainfall then falls once fields flood; a line is hopeless there. The trick is disarmingly simple: feed the model x² and x³ as if they were brand-new columns, and the **same straight-line machinery is suddenly free to curve**. We haven't changed the algorithm — only the ingredients we hand it. The danger is overdoing it:

A degree-1 line **underfits** (too rigid to follow the bend); a sensible degree captures the trend; a very high degree **overfits**, snaking through every noisy point and generalising badly. Choosing the degree is the core bias–variance trade-off of this method.

## Motivation

Real-world data rarely follows a straight line. Crop yield vs rainfall peaks at an optimal level then falls; Flipkart session duration vs conversion rate follows a U-curve. Polynomial regression extends linear regression by adding powers of the original features as new columns.

$$\hat{y} = \theta_0 + \theta_1 x + \theta_2 x^2 + \theta_3 x^3 + \cdots + \theta_d x^d$$

Despite the curved output, this is still a *linear model* because θ appear linearly. We simply create new features: x₁=x, x₂=x², x₃=x³, then apply ordinary linear regression.

## Bias–Variance Trade-off Illustrated

Degree d controls the bias-variance balance:

| Degree | Bias | Variance | Behaviour |
|---|---|---|---|
| d = 1 | High | Low | Underfits — misses the curve |
| d = 3–5 | Medium | Medium | Good generalisation (Goldilocks zone) |
| d = 10+ | Very low | Very high | Overfits — wiggles through every point |

## From Scratch with NumPy

In [ ]:
# Polynomial Regression — Bengaluru startup funding vs headcount growth
import numpy as np

# Months after Series A vs % headcount growth
X = np.array([1,2,3,4,5,6,7,8,9,10], dtype=float)
y = np.array([5,12,22,35,46,50,47,38,26,15], dtype=float)

def poly_features(x, degree):
    """Build [1, x, x², …, x^degree] matrix."""
    return np.column_stack([x**d for d in range(degree+1)])

def fit_poly(x, y, degree):
    Xp = poly_features(x, degree)
    theta = np.linalg.lstsq(Xp, y, rcond=None)[0]  # least-squares solve
    y_hat = Xp @ theta
    r2 = 1 - np.sum((y-y_hat)**2)/np.sum((y-y.mean())**2)
    return theta, r2

for deg in [1, 2, 3, 5, 8]:
    th, r2 = fit_poly(X, y, deg)
    print(f"degree={deg}  R²={r2:.4f}  {'OVERFIT' if r2>0.9999 and deg>5 else ''}")

# Degree-2 prediction for month 11
th2, _ = fit_poly(X, y, 2)
x_new = np.array([[1, 11, 121]])
print(f"\nMonth 11 forecast: {(x_new @ th2).item():.1f}% growth")

> **⚠ Read This Alongside the Interactive Widget Below**
>
> The degree-2 fit's month-11 extrapolation (−1.3%) looks alarming next to the R²=0.9343 fit quality — that's the point, not a contradiction. Degree 2 fits the observed months 1–10 well but is still a parabola, and this parabola happens to already be curving back downward by month 10; extrapolating just one more step continues that curve past zero. It's a genuine, real-numbers illustration of why extrapolating polynomial regression even slightly beyond the training range is risky, regardless of how good the in-range R² looks — a theme the widget below lets you explore directly.

## With scikit-learn Pipeline

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

X2d = X.reshape(-1,1)

# Compare degrees with Ridge regularisation + 5-fold CV
for deg in [1, 2, 3, 4]:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('scale', StandardScaler()),
        ('ridge', Ridge(alpha=1.0))
    ])
    scores = cross_val_score(pipe, X2d, y, cv=5, scoring='r2')
    print(f"deg={deg}  CV-R² = {scores.mean():.3f} ± {scores.std():.3f}")

## Try It — Drag the Degree Slider

The same Bengaluru startup-funding data from above, refit live at any degree from 1 to 9. Watch R² climb while the curve starts wiggling through every single point — the underfit → Goldilocks → overfit progression from the table above, in real time.

## Hyperparameters

| Parameter | Typical range | Effect |
|---|---|---|
| degree | 2–5 | Higher = more flexible, more overfit risk |
| Ridge α | 0.001 – 100 | Higher = stronger regularisation, prevents overfitting |
| interaction_only | True/False | Only cross-terms (no x², x³) when False |

## Beyond Fixed-Degree Polynomials: Generalized Additive Models

A single global polynomial degree is a blunt instrument — it forces the *same* curviness everywhere across the feature's range, when real relationships are often gently curved in some regions and sharply non-linear in others. **Generalized Additive Models (GAMs)** replace each feature's fixed-degree polynomial term with a flexible, data-driven **smooth function**:

$$\hat{y} = \beta_0 + f_1(x_1) + f_2(x_2) + \cdots + f_p(x_p)$$

Each fⱼ is a smooth curve (typically built from splines) fit to that one feature, learned from the data rather than fixed in advance as x² or x³. Critically, the model is still **additive** across features — no interactions by default — which keeps it almost as interpretable as linear regression: you can plot each fⱼ individually and read off exactly how that one feature affects the prediction, holding others fixed, something a black-box model can't offer directly.

In [ ]:
from pygam import LinearGAM, s
import numpy as np

# Swiggy delivery time: has a sharp non-linearity at rush hour that no single
# polynomial degree captures well across the WHOLE day
np.random.seed(1)
hour = np.random.uniform(0,24,500)
# calm most of the day, sharp spike 12-14h and 19-21h
delivery_min = 25 + 15*np.exp(-0.5*((hour-13)/1.2)**2) + 18*np.exp(-0.5*((hour-20)/1.5)**2) + np.random.normal(0,3,500)

gam = LinearGAM(s(0)).fit(hour.reshape(-1,1), delivery_min)  # s() = smooth spline term
for h in [9, 13, 16, 20]:
    pred = gam.predict([[h]])[0]
    print(f"  hour={h:2d}:00   predicted delivery time = {pred:.1f} min")
print(f"\nEffective degrees of freedom used: {gam.statistics_['edof']:.1f}  (learned automatically, not hand-picked)")

A single polynomial degree would need to be complex enough to capture both rush-hour spikes, but that same complexity would then also overfit the calm mid-morning and mid-afternoon periods — GAM's per-region smoothness sidesteps this entirely by letting the curve flex only where the data actually demands it.

> **💡 When to Reach for a GAM Instead of Polynomial Features**
>
> Polynomial regression (with Ridge) remains the right default for mild, single-region curvature — it's simpler, faster, and needs no extra library. Reach for a GAM specifically when a feature's relationship to the target changes *character* across its range (calm-then-spiky, as above) and you still need to preserve per-feature interpretability — the middle ground between a rigid polynomial and a fully black-box model like Random Forest.

> **🔗 Real-World Link — Food Delivery Time Prediction**
>
> 45,593 real Zomato-style delivery records let you test the obvious hypothesis — does straight-line distance predict delivery time? It barely correlates at all (r≈0); traffic and weather dominate instead. [See the case study →](https://statso.io/2023/01/02/food-delivery-time-prediction-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · A quadratic with PolynomialFeatures

`y = x²` exactly. Fit a degree-2 polynomial regression (`PolynomialFeatures` + `LinearRegression`) and store the prediction for x = 6 in `pred`.

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
x = np.arange(1, 9, dtype=float).reshape(-1, 1)
y = (x ** 2).ravel()
pred = None   # TODO


In [ ]:
try:
    check("6 squared is 36", abs(pred - 36) < 1e-6)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
x = np.arange(1, 9, dtype=float).reshape(-1, 1)
y = (x ** 2).ravel()
poly = PolynomialFeatures(2)
model = LinearRegression().fit(poly.fit_transform(x), y)
pred = model.predict(poly.transform([[6.0]]))[0]

```

</details>

### Exercise 2 · Medium · Choose the degree by cross-validation

Loop over degrees 1–5 with a `make_pipeline(PolynomialFeatures(d), LinearRegression())` and `cross_val_score(..., cv=5, scoring="r2")`. Store the degree with the best mean score in `best_degree` (the truth is quadratic).

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
rng = np.random.default_rng(3)
x = rng.uniform(-3, 3, 120).reshape(-1, 1)
y = 1 + 0.5 * x.ravel() - 2 * x.ravel() ** 2 + rng.normal(0, 1, 120)
best_degree = None   # TODO


In [ ]:
try:
    check("degree 2 (or a close 3) wins", best_degree in (2, 3))
    check("linear alone is not enough", best_degree != 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
rng = np.random.default_rng(3)
x = rng.uniform(-3, 3, 120).reshape(-1, 1)
y = 1 + 0.5 * x.ravel() - 2 * x.ravel() ** 2 + rng.normal(0, 1, 120)
scores = {d: cross_val_score(make_pipeline(PolynomialFeatures(d), LinearRegression()), x, y, cv=5, scoring="r2").mean() for d in range(1, 6)}
best_degree = max(scores, key=scores.get)

```

</details>

### Exercise 3 · Stretch · Measure overfitting

With only 15 noisy points, compare a degree-2 and a degree-10 fit. For each, compute the **gap** = train R² − test R² (test set given). Store the two gaps in `gap2` and `gap10`, and `overfits` = `gap10 > gap2`.

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(8)
f = lambda x: 2 + x - 0.5 * x ** 2
xtr = rng.uniform(-3, 3, 15).reshape(-1, 1); ytr = f(xtr.ravel()) + rng.normal(0, 1.0, 15)
xte = rng.uniform(-3, 3, 200).reshape(-1, 1); yte = f(xte.ravel()) + rng.normal(0, 1.0, 200)
gap2 = gap10 = overfits = None   # TODO


In [ ]:
try:
    check("both gaps computed", gap2 is not None and gap10 is not None)
    check("degree 10 overfits more", overfits is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(8)
f = lambda x: 2 + x - 0.5 * x ** 2
xtr = rng.uniform(-3, 3, 15).reshape(-1, 1); ytr = f(xtr.ravel()) + rng.normal(0, 1.0, 15)
xte = rng.uniform(-3, 3, 200).reshape(-1, 1); yte = f(xte.ravel()) + rng.normal(0, 1.0, 200)
def gap(d):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(xtr, ytr)
    return m.score(xtr, ytr) - m.score(xte, yte)
gap2, gap10 = gap(2), gap(10)
overfits = bool(gap10 > gap2)

```

A big train-minus-test gap is the signature of a model memorising noise instead of learning the pattern.

</details>

---
*Back to the course: **Machine Learning End To End → Polynomial Regression**.*